# Optiver TC1 Manual Baseline Notes

This notebook is an evidence artifact for the `tc1_from_scratch` human baseline.
It records dataset inspection and plain-language reasoning only.
The final bank-id mapping lives in `manual_selection_worksheet.md`, not in this notebook.


In [ ]:
import pandas as pd

train = pd.read_csv('~/.kaggle/competitions/optiver-trading-at-the-close/train.csv')
print('shape', train.shape)
print(train[['stock_id', 'date_id', 'seconds_in_bucket']].head())
print(train.isnull().mean().sort_values(ascending=False).head(8))


## Auction Missingness And Basic Price Cleanup

First-pass reasoning:

- `far_price` and `near_price` are noisy and incomplete, but they are too central to the auction snapshot to drop immediately
- keep the simple Optiver default fills without automatically escalating into grouped median repair
- keep the easy defaults and leave the grouped fill out of the first-pass baseline


In [ ]:
missing = train[['far_price', 'near_price', 'imbalance_size', 'matched_size']].isnull().mean()
print(missing)
print(train[['far_price', 'near_price', 'imbalance_size', 'matched_size']].describe())


## Core Microstructure, Temporal Dependence, And Priors

Second-pass reasoning:

- first-order imbalance and pairwise price disagreement features are the clearest reusable microstructure baseline
- one lag family plus one stock-prior family is enough for a believable first-pass manual baseline
- keep the simpler imbalance-ratio style feature and stop before the heavier grouped-median and higher-order interaction branches


In [ ]:
train['volume'] = train['ask_size'] + train['bid_size']
train['mid_price'] = (train['ask_price'] + train['bid_price']) / 2
train['liquidity_imbalance'] = (train['bid_size'] - train['ask_size']) / (train['bid_size'] + train['ask_size'])
train['matched_imbalance_ratio'] = train['imbalance_size'] / (train['matched_size'] + 1)
print(train[['volume', 'mid_price', 'liquidity_imbalance', 'matched_imbalance_ratio']].head())

ordered = train.sort_values(['stock_id', 'date_id', 'seconds_in_bucket'])
ordered['reference_price_lag_1'] = ordered.groupby('stock_id')['reference_price'].shift(1)
print(ordered[['stock_id', 'seconds_in_bucket', 'reference_price', 'reference_price_lag_1']].head())

stock_priors = ordered.groupby('stock_id')[['bid_size', 'ask_size', 'bid_price', 'ask_price']].agg(['median', 'std', 'min', 'max'])
print(stock_priors.head())


## Clock Features, Memory Discipline, And Manual Takeaways

- keep the simple far/near default fills
- keep the first-order imbalance block, pairwise price dislocations, and a simple imbalance ratio
- keep one lag family and one stock-prior strategy
- keep compact auction-clock features and final numeric downcasting
- stop before grouped-median repair, higher-order interaction blocks, and the broader rolling-window branch in this first-pass baseline
